# 자동차관리법 RAG 청킹 전략 비교 — 통합 실행본

기존 `data_prep.py`, `chunk_metrics.py`, `compare_chunking.py`, `visualize_comparison.py`를 하나로 합친 노트북입니다. 위에서 아래로 실행하면 청킹 비교, 성능 평가, CSV 저장, 그래프 생성까지 진행됩니다.

> `config.py`, `models.py`, `eval_data.py`와 자동차관리법 PDF는 기존 프로젝트 폴더에 함께 있어야 합니다.


## 0. 라이브러리 설치
처음 실행하는 환경에서만 주석을 해제하세요.

In [1]:
# %pip install -U pandas numpy matplotlib python-dotenv bert-score pymupdf4llm langchain-core langchain-text-splitters langchain-chroma sentence-transformers


## 1. 공통 라이브러리 및 프로젝트 설정

In [1]:
import os
import re
import time
import statistics
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymupdf4llm

from bert_score import score as bertscore
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

from config import CONFIG, EVAL_DIR, MD_CACHE_PATH, PDF_PATH
from eval_data import EVAL_DATA
from models import build_llm, build_retriever, build_vectorstore

load_dotenv()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

print("환경 변수 및 프로젝트 설정 로드 완료")
print(f"PDF_PATH: {PDF_PATH}")
print(f"EVAL_DIR: {EVAL_DIR}")


환경 변수 및 프로젝트 설정 로드 완료
PDF_PATH: C:\knu-kdt\project_team4\data\자동차관리법.pdf
EVAL_DIR: C:\knu-kdt\project_team4\eval


## 2. PDF 로드 및 청킹 함수

In [2]:



# "제34조(...)", "- 제27조의2(...)" 처럼 줄 시작에 오는 조문 번호를 감지
# chunk_metrics.py에서 조문 분할율(article split rate) 계산에도 재사용하므로 공개(public)로 둠
ARTICLE_SPLIT_PATTERN = re.compile(r"(?m)(?=^(?:- )?제\d+조(?:의\d+)?\()")


def load_markdown_text():
    """PDF -> 마크다운 변환 결과(캐시 활용)만 반환. compare_chunking.py에서 전략별로
    같은 md_text를 재사용(조문 분할율 계산 등)할 때 쓴다."""
    return _load_markdown()


def _load_markdown():
    assert os.path.exists(PDF_PATH), f"PDF 파일을 찾을 수 없습니다: {PDF_PATH} (경로를 확인하세요)"
    print(f"PDF 경로 확인 완료: {PDF_PATH}")

    if os.path.exists(MD_CACHE_PATH):
        with open(MD_CACHE_PATH, encoding="utf-8") as f:
            md_text = f.read()
        print(f"마크다운 캐시 재사용: {MD_CACHE_PATH} ({len(md_text)}자) — PDF 재변환 생략")
    else:
        md_text = pymupdf4llm.to_markdown(PDF_PATH)
        os.makedirs(os.path.dirname(MD_CACHE_PATH), exist_ok=True)
        with open(MD_CACHE_PATH, "w", encoding="utf-8") as f:
            f.write(md_text)
        print(f"마크다운 변환 완료 및 캐시 저장: {len(md_text)}자 -> {MD_CACHE_PATH}")
    return md_text


def load_and_split_character(chunk_size=None, chunk_overlap=None):
    """baseline: 문자 수 기준 분할 (B_Embedding_tuning과 동일한 방식).
    chunk_size/overlap을 생략하면 CONFIG의 baseline 값(500/50)을 사용."""
    chunk_size = chunk_size or CONFIG["chunk_size"]
    chunk_overlap = chunk_overlap if chunk_overlap is not None else CONFIG["chunk_overlap"]

    md_text = _load_markdown()
    docs = [Document(page_content=md_text, metadata={"source": PDF_PATH})]

    text_splitter = RecursiveCharacterTextSplitter.from_language(
        language=Language.MARKDOWN,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    splits = text_splitter.split_documents(docs)
    print(f"[character] 청킹 완료: {len(splits)}개 조각 (chunk_size={chunk_size}, overlap={chunk_overlap})")
    return splits


def load_and_split_article():
    """tuning: 조문("제O조") 단위로 먼저 분할 -> chunk_size를 넘는 조문만 하위 분할.
    법 조문처럼 구조화된 문서는 의미 단위(조문) 경계를 지키는 편이 검색 품질에 유리할 수 있음."""
    md_text = _load_markdown()

    raw_parts = ARTICLE_SPLIT_PATTERN.split(md_text)
    raw_parts = [p for p in raw_parts if p.strip()]
    print(f"[article] 조문 단위 1차 분할: {len(raw_parts)}개 조각")

    sub_splitter = RecursiveCharacterTextSplitter.from_language(
        language=Language.MARKDOWN,
        chunk_size=CONFIG["chunk_size"],
        chunk_overlap=CONFIG["chunk_overlap"],
    )

    docs = []
    for part in raw_parts:
        if len(part) <= CONFIG["chunk_size"]:
            docs.append(Document(page_content=part, metadata={"source": PDF_PATH}))
        else:
            # 조문 하나가 너무 길면(부칙, 별표 등) chunk_size 기준으로 추가 분할
            docs.extend(sub_splitter.split_documents(
                [Document(page_content=part, metadata={"source": PDF_PATH})]
            ))

    print(f"[article] 최종 청킹 완료: {len(docs)}개 조각 (초과 조문만 chunk_size={CONFIG['chunk_size']}로 하위 분할)")
    return docs


def load_and_split_markdown():
    """CONFIG['chunking_strategy']에 따라 baseline(character) 또는 tuning(article) 분할을 적용."""
    if CONFIG["chunking_strategy"] == "article":
        return load_and_split_article()
    return load_and_split_character()


## 3. 청킹 평가 지표 함수

In [3]:
"""
청킹 전략 비교에 쓰는 추가 성능지표 모음.
기존 compare_chunking.py는 BERTScore F1 / 응답시간 2개만 봤는데,
청킹 자체의 효과를 설명하려면 아래 3가지 축을 더 보는 게 좋음:

1) retrieval_hit()          — retrieval 단계: 정답 조문이 검색된 context 안에 실제로 들어왔는가
2) chunk_length_stats()     — 청크 구조: 청크 길이가 얼마나 고르게/치우쳐 분포하는가
3) article_split_rate()     — 청크 구조: 원본 조문이 청크 경계에서 잘려나간 비율
"""

# ground_truth 문장 속 "제34조", "제27조의2" 같은 조문 번호를 추출
_ARTICLE_REF_PATTERN = re.compile(r"제\d+조(?:의\d+)?")


def extract_articles(text):
    """텍스트에서 언급된 조문 번호 집합을 추출. 예: {'제34조', '제81조'}"""
    return set(_ARTICLE_REF_PATTERN.findall(text))


def retrieval_hit(context_text, ground_truth):
    """ground_truth가 인용하는 조문 중 하나라도 retriever가 가져온 context 안에
    등장하면 히트(1)로 판정. ground_truth에 조문 번호가 없으면 판정 불가(None)."""
    gt_articles = extract_articles(ground_truth)
    if not gt_articles:
        return None
    context_articles = extract_articles(context_text)
    return bool(gt_articles & context_articles)


def chunk_length_stats(splits):
    """청크(Document 리스트)의 문자 길이 분포 통계."""
    lengths = [len(d.page_content) for d in splits]
    if not lengths:
        return {"mean_len": 0, "std_len": 0, "min_len": 0, "max_len": 0, "median_len": 0}
    return {
        "mean_len": statistics.mean(lengths),
        "std_len": statistics.pstdev(lengths) if len(lengths) > 1 else 0.0,
        "min_len": min(lengths),
        "max_len": max(lengths),
        "median_len": statistics.median(lengths),
    }


def article_split_rate(md_text, splits):
    """원본 조문이 청크 하나에 온전히 담기지 못하고 여러 청크에 걸쳐 잘린 비율.

    md_text를 조문 단위로 재분할(data_prep.ARTICLE_SPLIT_PATTERN과 동일 기준)한 뒤,
    각 조문 전체 텍스트가 splits 중 어느 한 청크의 page_content 안에 완전히
    포함되는지 검사한다. 포함되면 안 잘린 것, 아니면 청크 경계에서 잘린 것으로 본다.
    (article 전략은 대부분 0%에 가깝고, character 전략은 청크 크기가 작을수록 높아짐)
    """
    articles = [p for p in ARTICLE_SPLIT_PATTERN.split(md_text) if p.strip()]
    if not articles:
        return {"split_rate": 0.0, "num_articles": 0, "num_split_articles": 0}

    chunk_texts = [d.page_content for d in splits]
    split_count = 0
    for article in articles:
        contained = any(article in chunk for chunk in chunk_texts)
        if not contained:
            split_count += 1

    return {
        "split_rate": split_count / len(articles),
        "num_articles": len(articles),
        "num_split_articles": split_count,
    }


## 4. 청킹 전략 비교 및 평가 함수

In [4]:



# 비교해볼 청킹 전략 — (전략 이름, 분할 함수). 자유롭게 추가/삭제하세요
STRATEGIES = [
    ("character_300", partial(load_and_split_character, chunk_size=300, chunk_overlap=50)),
    ("character_500 (baseline)", partial(load_and_split_character, chunk_size=500, chunk_overlap=50)),
    ("character_800", partial(load_and_split_character, chunk_size=800, chunk_overlap=100)),
    ("article (tuning)", load_and_split_article),
]


def evaluate_one_strategy(strategy_name, split_fn, llm, md_text):
    print(f"\n{'=' * 60}\n청킹 전략: {strategy_name}\n{'=' * 60}")

    splits = split_fn()
    collection_name = (
        CONFIG["run_name"] + "_" + strategy_name.split()[0] + "_" +
        CONFIG["embedding_model"].replace("/", "_").replace("-", "_")
    )
    vectorstore = build_vectorstore(splits, collection_name=collection_name)
    retriever = build_retriever(vectorstore)

    # rag_chain.build_rag_chain() 대신 prompt|llm을 직접 구성 — retriever가 가져온
    # context를 answer 생성 전에 미리 확보해서 retrieval hit 판정에 재사용하기 위함
    # (build_rag_chain을 그대로 쓰면 context가 체인 내부에 숨어서 꺼내 쓸 수 없음)
    prompt = ChatPromptTemplate.from_template(CONFIG["prompt_template"])
    answer_chain = prompt | llm | StrOutputParser()

    # --- 청크 구조 지표 (질문과 무관하게 전략 하나당 한 번만 계산) ---
    length_stats = chunk_length_stats(splits)
    split_stats = article_split_rate(md_text, splits)
    print(
        f"[{strategy_name}] 청크 길이: 평균 {length_stats['mean_len']:.0f}자 "
        f"(표준편차 {length_stats['std_len']:.0f}) / "
        f"조문 분할율: {split_stats['split_rate'] * 100:.1f}% "
        f"({split_stats['num_split_articles']}/{split_stats['num_articles']}개 조문)"
    )

    rows = []
    for item in EVAL_DATA:
        q = item["question"]

        start = time.time()
        docs = retriever.invoke(q)
        context = format_docs(docs)
        answer = answer_chain.invoke({"context": context, "question": q})
        elapsed = time.time() - start

        hit = retrieval_hit(context, item["ground_truth"])

        rows.append({
            "chunking_strategy": strategy_name,
            "num_chunks": len(splits),
            "chunk_mean_len": length_stats["mean_len"],
            "chunk_std_len": length_stats["std_len"],
            "article_split_rate": split_stats["split_rate"],
            "category": item["category"],
            "question": q,
            "ground_truth": item["ground_truth"],
            "answer": answer,
            "response_time_sec": round(elapsed, 3),
            "retrieval_hit": hit,  # True/False, ground_truth에 조문 번호가 없으면 None
            "context_char_len": len(context),
        })
        hit_label = "적중" if hit else ("미확인" if hit is None else "실패")
        print(f"[{elapsed:.2f}s][retrieval {hit_label}] Q: {q}\n -> A: {answer[:80]}...\n")

    df = pd.DataFrame(rows)

    P, R, F1 = bertscore(
        df["answer"].tolist(),
        df["ground_truth"].tolist(),
        model_type="klue/bert-base",
        num_layers=12,
        lang="ko",
        verbose=False,
    )
    df["bertscore_precision"] = P.tolist()
    df["bertscore_recall"] = R.tolist()
    df["bertscore_f1"] = F1.tolist()

    # retrieval_hit이 None(=ground_truth에 조문 번호 없음)인 행은 hit rate 계산에서 제외
    judged = df["retrieval_hit"].dropna()
    hit_rate = judged.mean() if len(judged) > 0 else float("nan")

    print(
        f"[{strategy_name}] 평균 BERTScore F1: {df['bertscore_f1'].mean():.4f} "
        f"(P={df['bertscore_precision'].mean():.4f} / R={df['bertscore_recall'].mean():.4f}) "
        f"/ 평균 응답시간: {df['response_time_sec'].mean():.2f}초 "
        f"/ 청크 수: {len(splits)} "
        f"/ Retrieval Hit Rate: {hit_rate * 100:.1f}% ({judged.sum()}/{len(judged)})"
    )
    return df


def run_experiment():
    load_dotenv()
    print("환경 변수 로드 완료")

    llm = build_llm()  # LLM은 한 번만 로드해서 모든 전략에 재사용
    md_text = load_markdown_text()  # 전략마다 동일한 원문을 재사용 (조문 분할율 계산용)

    all_dfs = []
    failed = []
    for name, fn in STRATEGIES:
        try:
            all_dfs.append(evaluate_one_strategy(name, fn, llm, md_text))
        except Exception as e:
            print(f"\n[{name}] 실패 — 건너뛰고 계속 진행: {type(e).__name__}: {e}\n")
            failed.append(name)
    if failed:
        print(f"\n실패해서 제외된 전략: {failed}")
    df_all = pd.concat(all_dfs, ignore_index=True)

    os.makedirs(EVAL_DIR, exist_ok=True)

    detail_path = os.path.join(EVAL_DIR, "results_C_chunking_comparison_all.csv")
    df_all.to_csv(detail_path, index=False, encoding="utf-8-sig")
    print(f"\n전체 상세 결과 저장 완료: {detail_path}")

    # retrieval_hit(True/False/None)을 숫자로 바꿔서 groupby 집계가 가능하게 함
    df_all["retrieval_hit_numeric"] = df_all["retrieval_hit"].map({True: 1.0, False: 0.0})

    summary = (
        df_all.groupby("chunking_strategy")
        .agg(
            mean_bertscore_f1=("bertscore_f1", "mean"),
            mean_bertscore_precision=("bertscore_precision", "mean"),
            mean_bertscore_recall=("bertscore_recall", "mean"),
            mean_response_time_sec=("response_time_sec", "mean"),
            num_chunks=("num_chunks", "first"),
            chunk_mean_len=("chunk_mean_len", "first"),
            chunk_std_len=("chunk_std_len", "first"),
            article_split_rate=("article_split_rate", "first"),
            retrieval_hit_rate=("retrieval_hit_numeric", "mean"),
        )
        .sort_values("mean_bertscore_f1", ascending=False)
        .reset_index()
    )
    summary_path = os.path.join(EVAL_DIR, "results_C_chunking_comparison_summary.csv")
    summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

    print("\n=== 청킹 전략 비교 결과 (F1 높은 순) ===")
    print(summary.to_string(index=False))
    print(f"\n요약 결과 저장 완료: {summary_path}")

    best = summary.iloc[0]
    print(f"\n최고 F1 청킹 전략: {best['chunking_strategy']} (F1={best['mean_bertscore_f1']:.4f})")

    return df_all, summary



## 5. 전체 실험 실행
LLM 호출과 BERTScore 계산으로 시간이 오래 걸릴 수 있습니다.

In [ ]:
df_all, summary = run_experiment()
display(summary)


환경 변수 로드 완료


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM 로드 완료 (로컬 GPU, 4bit): Qwen/Qwen2.5-7B-Instruct
PDF 경로 확인 완료: C:\knu-kdt\project_team4\data\자동차관리법.pdf
마크다운 변환 완료 및 캐시 저장: 209431자 -> C:\knu-kdt\project_team4\C_Chunking_tuning_py\.cache\자동차관리법.md

청킹 전략: character_300
PDF 경로 확인 완료: C:\knu-kdt\project_team4\data\자동차관리법.pdf
마크다운 캐시 재사용: C:\knu-kdt\project_team4\C_Chunking_tuning_py\.cache\자동차관리법.md (209431자) — PDF 재변환 생략
[character] 청킹 완료: 920개 조각 (chunk_size=300, overlap=50)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[986.21s][retrieval 실패] Q: 비사업용 승용자동차의 최초 등록 후 정기검사(종합검사) 유효기간은 어떻게 되며, 검사 기간을 경과했을 때 부과되는 과태료 기준은 무엇인가요?
 -> A: 문맥에서 직접적으로 비사업용 승용자동차의 정기검사(종합검사)의 유효기간과 검사 기간을 경과했을 때 부과되는 과태료 기준에 대한 정보가 제공되어 ...



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[662.91s][retrieval 실패] Q: 중고차를 매매할 때 매매업자가 발급해 주는 '자동차 성능·상태 점검기록부'의 보증 기간과 보증 범위는 법적으로 어떻게 규정되어 있나요?
 -> A: 현재 제공된 문맥에서 '자동차 성능·상태 점검기록부'와 관련된 보증 기간과 보증 범위에 대한 법적 규정을 찾을 수 없습니다. 문맥에서는 자동차 ...



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[899.29s][retrieval 적중] Q: 자동차 등록번호판을 고의로 가리거나 알아보기 곤란하게 만든 경우, 자동차관리법에 따라 어떤 제재나 벌금을 받게 되나요?
 -> A: 자동차 등록번호판을 고의로 가리거나 알아보기 곤란하게 만든 경우, 자동차관리법에 따르면 다음과 같은 제재나 벌금을 받게 됩니다:

1. 제10조...



## 6. 결과 시각화 함수

In [ ]:
# Windows 기본 한글 폰트 (없으면 한글이 네모(□)로 깨짐)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

DETAIL_PATH = os.path.join(EVAL_DIR, "results_C_chunking_comparison_all.csv")
SUMMARY_PATH = os.path.join(EVAL_DIR, "results_C_chunking_comparison_summary.csv")
OUTPUT_PATH = os.path.join(EVAL_DIR, "chunking_comparison.png")


def _bar_with_labels(ax, labels, values, color, title, ylabel, fmt="{:.3f}", ylim_pad=1.2):
    bars = ax.bar(labels, values, color=color)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    max_val = max(values) if len(values) and max(values) > 0 else 1
    ax.set_ylim(0, max_val * ylim_pad)
    plt.setp(ax.get_xticklabels(), rotation=15, ha="right", fontsize=9)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), fmt.format(value),
                 ha="center", va="bottom", fontsize=9)
    return bars


def visualize_results():
    df_all = pd.read_csv(DETAIL_PATH)
    summary = pd.read_csv(SUMMARY_PATH).sort_values("mean_bertscore_f1", ascending=False)

    labels = summary["chunking_strategy"].tolist()

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))

    # (1,1) BERTScore F1
    _bar_with_labels(
        axes[0, 0], labels, summary["mean_bertscore_f1"], "#4C72B0",
        "청킹 전략별 평균 BERTScore F1", "BERTScore F1",
    )

    # (1,2) Precision vs Recall — 묶은 막대
    ax = axes[0, 1]
    x = np.arange(len(labels))
    width = 0.35
    bars_p = ax.bar(x - width / 2, summary["mean_bertscore_precision"], width, color="#4C72B0", label="Precision")
    bars_r = ax.bar(x + width / 2, summary["mean_bertscore_recall"], width, color="#8CB6E8", label="Recall")
    ax.set_title("청킹 전략별 BERTScore Precision / Recall")
    ax.set_ylabel("Score")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
    ax.legend(fontsize=8)
    for bars in (bars_p, bars_r):
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{bar.get_height():.3f}",
                     ha="center", va="bottom", fontsize=7.5)

    # (1,3) Retrieval Hit Rate — 정답 조문이 검색된 context에 포함된 비율
    _bar_with_labels(
        axes[0, 2], labels, summary["retrieval_hit_rate"], "#C44E52",
        "청킹 전략별 Retrieval Hit Rate", "Hit Rate", fmt="{:.0%}",
    )

    # (2,1) 평균 응답시간
    _bar_with_labels(
        axes[1, 0], labels, summary["mean_response_time_sec"], "#DD8452",
        "청킹 전략별 평균 응답시간 (초)", "초", fmt="{:.1f}s",
    )

    # (2,2) 청크 길이 분포 — 평균 ± 표준편차
    ax = axes[1, 1]
    bars = ax.bar(labels, summary["chunk_mean_len"], color="#55A868",
                   yerr=summary["chunk_std_len"], capsize=5)
    ax.set_title("청킹 전략별 청크 길이 (평균 ± 표준편차)")
    ax.set_ylabel("문자 수")
    ax.set_ylim(0, (summary["chunk_mean_len"] + summary["chunk_std_len"]).max() * 1.25)
    plt.setp(ax.get_xticklabels(), rotation=15, ha="right", fontsize=9)
    for bar, value in zip(bars, summary["chunk_mean_len"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{value:.0f}",
                 ha="center", va="bottom", fontsize=9)

    # (2,3) 조문 분할율 — 원본 조문이 청크 경계에서 잘린 비율 (낮을수록 좋음)
    _bar_with_labels(
        axes[1, 2], labels, summary["article_split_rate"], "#8172B2",
        "청킹 전략별 조문 분할율 (낮을수록 좋음)", "분할율", fmt="{:.0%}",
    )

    fig.suptitle("청킹 전략 비교 (자동차관리법 RAG)", fontsize=16)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(OUTPUT_PATH, dpi=150)
    print(f"그래프 저장 완료: {OUTPUT_PATH}")
    plt.show()



NameError: name 'plt' is not defined

## 7. 비교 그래프 생성

In [ ]:
visualize_results()
